In [1]:
import pymupdf
import pandas as pd
from pathlib import Path
import os
import re

In [2]:
DATA_DIR = Path("data")
ARTICLE_PATTERN = re.compile(r"\bArticle\s+\d+")

In [3]:
def format_date(date_str):
    return pd.to_datetime(date_str.replace("D:", "")[:8], format="%Y%m%d", errors="coerce")

def count_ratio_alnum_chars(content):
    content_length = len(content) if len(content) > 0 else 1
    nb_alnum_chars = 0
    nb_al_chars = 0
    for char in content:
        if char.isalnum():
            nb_alnum_chars += 1
            if char.isalpha():
                nb_al_chars += 1
    ratio_alnum_chars = nb_alnum_chars / content_length
    ratio_al_chars = nb_al_chars / content_length
    ratio_num_chars = (nb_alnum_chars - nb_al_chars) / content_length
    return ratio_alnum_chars, ratio_al_chars, ratio_num_chars

In [4]:
# create df to calculate stats on it

stats = []

for file in DATA_DIR.iterdir():
    try:
        with pymupdf.open(file) as f:
            creation_date = format_date(f.metadata['creationDate'])
            nb_pages = f.page_count
            for no_page, page in enumerate(f):
                page_content = page.get_text()
                # TODO : use https://pymupdf.readthedocs.io/en/latest/page.html#Page.get_textpage_ocr and https://stackoverflow.com/questions/76834972/how-can-i-run-pytesseract-python-library-in-ubuntu-22-04
                # FIXME : pas de tres bons resultats avec cette methode...
                #if len(page_content) == 0:
                #    text_page = page.get_textpage_ocr(flags=0, language='fra', dpi=600, full=True)
                #    page_content = page.get_text(textpage=text_page)
                #    with open(os.path.join(file.stem + ".txt"), "a") as content:
                #        content.write(page_content)
                article_matches = ARTICLE_PATTERN.findall(page_content)
                ratio_alnum, ratio_al, ratio_num = count_ratio_alnum_chars(page_content)
                stats.append({
                    "filename_page_no": file.name + "_" + str(no_page),
                    "filename": file.name,
                    "creation_date": creation_date,
                    "nb_pages": nb_pages,
                    "content_length": len(page_content),
                    "ratio_alnum": ratio_alnum,
                    "ratio_al": ratio_al,
                    "ratio_num": ratio_num,
                    "article_occ_count": len(article_matches)
                })
    except Exception as e:
        # FIXME : uniquement print error ou ajouter ligne "vide" dans df (dans tous les cas mettre dans les logs)
        print(f"Error while reading {file}: {e}")

df_stats = pd.DataFrame(stats)

In [5]:
# stats on volume

df_stats_by_file = (
    df_stats.groupby("filename", as_index=False).agg(
          creation_date=pd.NamedAgg(column="creation_date", aggfunc="first"),
          nb_pages=pd.NamedAgg(column="nb_pages", aggfunc="first"),
          content_length=pd.NamedAgg(column="content_length", aggfunc="sum"),
          ratio_alnum=pd.NamedAgg(column="ratio_alnum", aggfunc="mean"),
          ratio_al=pd.NamedAgg(column="ratio_al", aggfunc="mean"),
          ratio_num=pd.NamedAgg(column="ratio_num", aggfunc="mean"),
          article_occ_count=pd.NamedAgg(column="article_occ_count", aggfunc="sum")
      )
)

nb_files = df_stats_by_file["filename"].nunique()
nb_total_pages = df_stats_by_file["nb_pages"].sum()
nb_total_char = df_stats_by_file["content_length"].sum()

assert(len(df_stats_by_file) == sum(1 for f in os.listdir(DATA_DIR)))

print(f"There are {nb_files} files in total.")
print(f"There are {nb_total_pages} pages in total.")
print(f"There are {nb_total_char} characters in total.")

There are 295 files in total.
There are 3207 pages in total.
There are 7851804 characters in total.


In [6]:
# TODO : analyser distributions ci-dessous
df_stats_by_file

,filename,creation_date,nb_pages,content_length,ratio_alnum,ratio_al,ratio_num,article_occ_count
0,en_2AKqesrYsNz5awErW_LEX_8.3.1.pdf,2002-12-11,18,28706,0.717563,0.689742,0.027821,0
1,en_2CkeMAKnJHgbS6hhd_LEX_4.6.0.2.pdf,2025-03-25,5,11010,0.745351,0.730118,0.015233,11
2,en_2DfZ6Zu4kEK5kp3gG_LEX_1.3.5.pdf,2019-08-15,7,16433,0.775664,0.749681,0.025983,12
3,en_2Qu7vQ5eC9ZhKc6hq_LEX_5.7.0.2.pdf,2021-09-24,12,0,0.000000,0.000000,0.000000,0
4,en_2WgDd3ZztEQP6cZJ6_LEX_4.2.5.pdf,2025-04-15,3,6095,0.777149,0.733872,0.043277,12
...,...,...,...,...,...,...,...,...
290,fr_xLxaXn4QFoNKxidFR_LEX_1.1.17.pdf,2025-04-14,12,23947,0.811211,0.778858,0.032353,0
291,fr_xNnk9WbJmWemDX3YJ_LEX_4.1.2.pdf,2025-03-10,2,4845,0.783543,0.766336,0.017206,3
292,fr_yDEG42FNmHfYxT6rg_LEX_6.1.5.pdf,2025-03-13,3,9173,0.788348,0.761777,0.026572,9
293,fr_ySZ4jwt54JonrnsdT_LEX_2.2.1.pdf,2023-04-17,3,6928,0.754688,0.714815,0.039874,10


In [7]:
# TODO : analyser distributions ci-dessous

df_stats_by_page = df_stats[
    ["filename_page_no", "content_length", "ratio_alnum", "ratio_al", "ratio_num", "article_occ_count"]
]
df_stats_by_page

,filename_page_no,content_length,ratio_alnum,ratio_al,ratio_num,article_occ_count
0,fr_cpp7qMgRdYrvWRWWe_LEX_1.7.1.pdf_0,2602,0.780938,0.766718,0.014220,4
1,fr_cpp7qMgRdYrvWRWWe_LEX_1.7.1.pdf_1,1858,0.766416,0.736276,0.030140,4
2,en_9nKZfK3P6cknvXyRs_LEX_2.6.4.pdf_0,2890,0.800000,0.777509,0.022491,4
3,en_9nKZfK3P6cknvXyRs_LEX_2.6.4.pdf_1,2891,0.796956,0.776202,0.020754,4
4,en_oZkqFqxSzX6nS4pFQ_LEX_2.6.1.pdf_0,2287,0.804110,0.782247,0.021863,5
...,...,...,...,...,...,...
3202,en_JJpbmCJ964DfEMJ9k_LEX_2.4.0.2.pdf_0,2171,0.793183,0.774758,0.018425,1
3203,en_JJpbmCJ964DfEMJ9k_LEX_2.4.0.2.pdf_1,2796,0.791488,0.786838,0.004649,1
3204,en_JJpbmCJ964DfEMJ9k_LEX_2.4.0.2.pdf_2,2311,0.794461,0.784509,0.009952,0
3205,en_JJpbmCJ964DfEMJ9k_LEX_2.4.0.2.pdf_3,2977,0.796775,0.785018,0.011757,3


In [8]:
# TODO : analyser quels documents sont tres courts et lequels sont tres longs
# TODO : calculer longueur moyenne des mots, proportion de mots inconnus (via dictionnaire, pour savoir si noms propres ou OCR raté ou autres langues) et ratio mots uniques / total (trop élevé = bruit)
# TODO : cleaner les contenus (tableaux et autres)
# TODO : deduplication et redondance -> comparer similarite cosine embeddings pour etre sur de pas avoir de duplications dans la vector db (surement le cas puisque pdf parfois a double)

In [10]:
# FIXME : base sur https://medium.com/@alice.yang_10652/extract-text-from-images-and-scanned-pdfs-with-python-2087cb1e0a7b

from spire.pdf import *
from spire.ocr import *

SCANNED_DATA_DIR = Path('text_from_scanned_pdfs')
os.makedirs(SCANNED_DATA_DIR, exist_ok=True)

scanned_files = df_stats_by_file['filename'][df_stats_by_file['content_length'] == 0].values

def convert_page_to_image(file, page_index):
    return file.SaveAsImage(page_index)

def recognize_text_from_image(image_name, language, model_path):
    scanner = OcrScanner()
    configure_options = ConfigureOptions()
    configure_options.Language = language
    configure_options.ModelPath = model_path
    scanner.ConfigureDependencies(configure_options)

    scanner.Scan(str(image_name))
    data = scanner.Text.ToString()
    return data

for file in scanned_files:
    filepath = os.path.join(DATA_DIR, file)
    pdf = PdfDocument()
    pdf.LoadFromFile(str(filepath))

    filename = os.path.splitext(os.path.basename(filepath))[0]
    text_filename = filename + ".txt"

    image_dir = os.path.join(SCANNED_DATA_DIR, filename)
    os.makedirs(image_dir, exist_ok=True)

    with open(os.path.join(SCANNED_DATA_DIR, text_filename), 'a', encoding='utf-8') as writer:
        for page_index in range(pdf.Pages.Count):
            image = convert_page_to_image(pdf, page_index)
            image_name = os.path.join(image_dir, f"{page_index}.png")
            image.Save(image_name)
            recognized_text = recognize_text_from_image(image_name, 'French', r'ocr_model')
            #writer.write(f'Page {page_index + 1}:\n')
            writer.write(recognized_text)
            #writer.write('\n\n')